# Create training subset


In [ ]:
import hashlib
from pathlib import Path

import numpy as np
import pandas as pd
import your
from sklearn.model_selection import train_test_split
from tqdm.auto import tqdm


## Set input and output paths


In [ ]:
filterbank_file_path = Path('/hercules/results/akazantsev/filterbank_files/B0531+21/B0531+21_59000_48386.fil')
segment_labels_path = Path('/u/akazantsev/ml_dl_for_rfi_mitigation/segments_index_B0531+21_59000_48386.csv')
output_dir = Path('/u/akazantsev/ml_dl_for_rfi_mitigation/runs/punch_rfi_reproduction')
expected_filterbank_md5 = '3cf053e6bc46911eada8d19bcb591bf5'

if not filterbank_file_path.is_file():
    raise FileNotFoundError(f'Filterbank file not found: {filterbank_file_path}')
if not segment_labels_path.is_file():
    raise FileNotFoundError(f'Segment labels file not found: {segment_labels_path}')

output_dir.mkdir(parents=True, exist_ok=True)

base_name = filterbank_file_path.stem
channels_path = output_dir / f'{base_name}_training_channels.npy'
metadata_path = output_dir / f'{base_name}_training_metadata.csv'
split_path = output_dir / f'{base_name}_training_split_indices.npz'
output_paths = (channels_path, metadata_path, split_path)

# Stop before an existing result can be overwritten.
for output_path in output_paths:
    if output_path.exists():
        raise FileExistsError(f'Output already exists: {output_path}')


## Define labels and annotations


In [ ]:
n_channels = 256
n_time_samples = 256
n_per_training_class = 10_000
random_state = 42

NBRFI = 'NBRFI'
BGN = 'BGN'
BGN_RFISEG = 'BGN_RFISEG'

frequency_range = {
    'B0531+21_59000_48386.fil': np.linspace(1530.0, 1210.0, n_channels)
}

rfi_blocks = {
    'B0531+21_59000_48386.fil': {
        1: (204, 1527), 2: (2023, 4436), 3: (4437, 4953),
        4: (6701, 7631), 5: (42546, 45015), 6: (45622, 45705),
        7: (46186, 46927), 8: (46928, 47303), 9: (48686, 48831),
        10: (48832, 49591)
    }
}

segments_rfi_indexes = {
    'B0531+21_59000_48386.fil': {
        1: [[230, 235], [239, 241]], 2: [230, 256],
        3: [230, 256], 4: [[230, 235], [239, 243]],
        5: [[189, 195], [199, 202], [205, 209]],
        6: [[189, 195], [199, 202], [206, 210]],
        7: [[190, 193], [199, 202], [206, 210]],
        8: [[188, 194], [198, 203], [206, 210]],
        9: [[188, 194], [198, 203], [206, 210]],
        10: [[188, 194], [198, 203], [206, 210]]
    }
}


## Define data functions


In [ ]:
def md5sum(path, block_size=1024 * 1024):
    digest = hashlib.md5()
    with path.open('rb') as stream:
        for block in iter(lambda: stream.read(block_size), b''):
            digest.update(block)
    return digest.hexdigest()


def get_segment_data(filterbank_file, segment_index):
    steps = range(0, filterbank_file.your_header.nspectra, n_time_samples)
    return filterbank_file.get_data(
        nstart=steps[segment_index], nsamp=n_time_samples
    ).T


def compute_stats(x, axis=None):
    x = np.asarray(x)
    mean = x.mean(axis=axis, keepdims=True)
    std = x.std(axis=axis, keepdims=True)

    centered = x - mean
    m3 = (centered ** 3).mean(axis=axis, keepdims=True)
    m4 = (centered ** 4).mean(axis=axis, keepdims=True)

    std_safe = np.where(std == 0, 1.0, std)
    skew = m3 / (std_safe ** 3)
    kurt = m4 / (std_safe ** 4) - 3.0

    return mean.squeeze(), std.squeeze(), skew.squeeze(), kurt.squeeze()


def normalize_per_channel(segment):
    segment = np.asarray(segment)
    channel_min = segment.min(axis=1, keepdims=True)
    channel_max = segment.max(axis=1, keepdims=True)
    denominator = channel_max - channel_min
    denominator_safe = np.where(denominator == 0, 1.0, denominator)
    return (segment - channel_min) / denominator_safe * 256.0


def compute_ratio(channel_values, segment_value):
    channel_values = np.asarray(channel_values, dtype=float)
    if segment_value == 0:
        return np.full_like(channel_values, np.nan, dtype=float)
    return channel_values / segment_value


def get_rfi_block_id(segment_index, blocks):
    for block_id, (start, stop) in blocks.items():
        if start <= segment_index <= stop:
            return block_id
    return None


def get_rfi_channel_ranges(block_id, channel_ranges):
    ranges = channel_ranges[block_id]
    if isinstance(ranges[0], (list, tuple, np.ndarray)):
        return [(int(start), int(stop)) for start, stop in ranges]
    return [(int(ranges[0]), int(ranges[1]))]


def build_rfi_mask(channel_ranges):
    mask = np.zeros(n_channels, dtype=bool)
    for start, stop in channel_ranges:
        start = max(0, start)
        stop = min(n_channels - 1, stop)
        if start <= stop:
            mask[start:stop + 1] = True
    return mask


## Select training channels


In [ ]:
file_hash = md5sum(filterbank_file_path)
if file_hash != expected_filterbank_md5:
    raise ValueError(f'Unexpected filterbank MD5: {file_hash}')

segment_labels = pd.read_csv(segment_labels_path)
required_columns = {'global_index', 'segment_index', 'label'}
if not required_columns.issubset(segment_labels.columns):
    raise ValueError('Segment labels file has an unexpected schema')

file_key = filterbank_file_path.name
frequencies = frequency_range[file_key]
blocks = rfi_blocks[file_key]
channel_ranges = segments_rfi_indexes[file_key]

clean_segments = segment_labels.loc[
    segment_labels['label'].isna(), ['global_index', 'segment_index', 'label']
].reset_index(drop=True)
rfi_segments = segment_labels.loc[
    segment_labels['label'].eq('bright_NBRFI'),
    ['global_index', 'segment_index', 'label']
].reset_index(drop=True)

if len(clean_segments) * n_channels < n_per_training_class:
    raise ValueError('Not enough BGN channels are available')

rfi_candidates = []
bgn_rfiseg_count = 0
for row in rfi_segments.itertuples(index=False):
    block_id = get_rfi_block_id(int(row.segment_index), blocks)
    if block_id is None:
        continue

    rfi_mask = build_rfi_mask(get_rfi_channel_ranges(block_id, channel_ranges))
    bgn_rfiseg_count += int((~rfi_mask).sum())
    for channel_index in np.flatnonzero(rfi_mask):
        rfi_candidates.append({
            'global_index': int(row.global_index),
            'segment_index': int(row.segment_index),
            'channel_index': int(channel_index),
            'label': NBRFI,
            'original_segment_label': row.label
        })

if len(rfi_candidates) < n_per_training_class:
    raise ValueError('Not enough NBRFI channels are available')

rng = np.random.default_rng(random_state)

clean_channel_positions = rng.choice(
    len(clean_segments) * n_channels,
    size=n_per_training_class,
    replace=False
)
clean_segment_positions = clean_channel_positions // n_channels
clean_channel_indices = clean_channel_positions % n_channels

bgn_candidates = pd.DataFrame({
    'global_index': clean_segments['global_index'].to_numpy()[clean_segment_positions],
    'segment_index': clean_segments['segment_index'].to_numpy()[clean_segment_positions],
    'channel_index': clean_channel_indices,
    'label': BGN,
    'original_segment_label': clean_segments['label'].to_numpy()[clean_segment_positions]
})

rfi_positions = rng.choice(len(rfi_candidates), size=n_per_training_class, replace=False)
nbrfi_candidates = pd.DataFrame([rfi_candidates[position] for position in rfi_positions])

selected_channels = pd.concat([bgn_candidates, nbrfi_candidates], ignore_index=True)
selected_channels = selected_channels.iloc[
    rng.permutation(len(selected_channels))
].reset_index(drop=True)
selected_channels.insert(0, 'sample_index', np.arange(len(selected_channels)))

label_counts = selected_channels['label'].value_counts().to_dict()
if label_counts != {BGN: n_per_training_class, NBRFI: n_per_training_class}:
    raise RuntimeError(f'Unexpected training label counts: {label_counts}')

print('Candidate labels:', BGN, NBRFI, BGN_RFISEG)
print('BGN_RFISEG candidates:', bgn_rfiseg_count)
print('Selected channels:', len(selected_channels))
print('Selected labels:', label_counts)


## Read selected channels and calculate features


In [ ]:
filterbank_file = your.Your(str(filterbank_file_path))
training_channels = np.empty(
    (len(selected_channels), n_time_samples), dtype=np.float32
)
metadata_rows = [None] * len(selected_channels)

for segment_index, group in tqdm(
    selected_channels.groupby('segment_index', sort=True),
    total=selected_channels['segment_index'].nunique(),
    desc='Reading selected segments'
):
    segment = np.asarray(get_segment_data(filterbank_file, int(segment_index)))
    if segment.shape != (n_channels, n_time_samples):
        raise ValueError(f'Unexpected segment shape: {segment.shape}')

    mean_o_segment, std_o_segment, skew_o_segment, kurt_o_segment = compute_stats(segment)
    normalized_segment = normalize_per_channel(segment)
    mean_n_segment, std_n_segment, skew_n_segment, kurt_n_segment = compute_stats(normalized_segment)

    mean_o_channel, std_o_channel, skew_o_channel, kurt_o_channel = compute_stats(segment, axis=1)
    mean_n_channel, std_n_channel, skew_n_channel, kurt_n_channel = compute_stats(normalized_segment, axis=1)

    mean_o_ratio = compute_ratio(mean_o_channel, mean_o_segment)
    std_o_ratio = compute_ratio(std_o_channel, std_o_segment)
    skew_o_ratio = compute_ratio(skew_o_channel, skew_o_segment)
    kurt_o_ratio = compute_ratio(kurt_o_channel, kurt_o_segment)
    mean_n_ratio = compute_ratio(mean_n_channel, mean_n_segment)
    std_n_ratio = compute_ratio(std_n_channel, std_n_segment)
    skew_n_ratio = compute_ratio(skew_n_channel, skew_n_segment)
    kurt_n_ratio = compute_ratio(kurt_n_channel, kurt_n_segment)

    for row in group.itertuples(index=False):
        channel_index = int(row.channel_index)
        sample_index = int(row.sample_index)

        if row.label == NBRFI:
            block_id = get_rfi_block_id(int(segment_index), blocks)
            expected_labels = np.where(
                build_rfi_mask(get_rfi_channel_ranges(block_id, channel_ranges)),
                NBRFI,
                BGN_RFISEG
            )
            if expected_labels[channel_index] != NBRFI:
                raise RuntimeError('Selected NBRFI channel is outside an RFI range')

        training_channels[sample_index] = segment[channel_index].astype(np.float32)
        metadata_rows[sample_index] = {
            'sample_index': sample_index,
            'global_index': int(row.global_index),
            'segment_index': int(segment_index),
            'channel_index': channel_index,
            'frequency': float(frequencies[channel_index]),
            'mean_o': float(mean_o_channel[channel_index]),
            'std_o': float(std_o_channel[channel_index]),
            'skew_o': float(skew_o_channel[channel_index]),
            'kurt_o': float(kurt_o_channel[channel_index]),
            'mean_n': float(mean_n_channel[channel_index]),
            'std_n': float(std_n_channel[channel_index]),
            'skew_n': float(skew_n_channel[channel_index]),
            'kurt_n': float(kurt_n_channel[channel_index]),
            'mean_o_ratio': float(mean_o_ratio[channel_index]),
            'std_o_ratio': float(std_o_ratio[channel_index]),
            'skew_o_ratio': float(skew_o_ratio[channel_index]),
            'kurt_o_ratio': float(kurt_o_ratio[channel_index]),
            'mean_n_ratio': float(mean_n_ratio[channel_index]),
            'std_n_ratio': float(std_n_ratio[channel_index]),
            'skew_n_ratio': float(skew_n_ratio[channel_index]),
            'kurt_n_ratio': float(kurt_n_ratio[channel_index]),
            'label': row.label,
            'original_segment_label': row.original_segment_label
        }

metadata = pd.DataFrame(metadata_rows)
feature_columns = [
    'mean_o', 'std_o', 'skew_o', 'kurt_o',
    'mean_n', 'std_n', 'skew_n', 'kurt_n',
    'mean_o_ratio', 'std_o_ratio', 'skew_o_ratio', 'kurt_o_ratio',
    'mean_n_ratio', 'std_n_ratio', 'skew_n_ratio', 'kurt_n_ratio'
]

if training_channels.shape != (2 * n_per_training_class, n_time_samples):
    raise RuntimeError('Training channel array has an unexpected shape')
if metadata['sample_index'].to_numpy().tolist() != list(range(len(metadata))):
    raise RuntimeError('Metadata rows do not match channel rows')
if len(feature_columns) != 16:
    raise RuntimeError('Statistical feature count is not 16')


## Create the report split and save outputs


In [ ]:
targets = (metadata['label'].to_numpy() == NBRFI).astype(np.int64)
sample_indices = metadata['sample_index'].to_numpy()

train_indices, temporary_indices = train_test_split(
    sample_indices,
    test_size=0.2,
    random_state=random_state,
    stratify=targets
)
temporary_targets = targets[temporary_indices]
validation_indices, test_indices = train_test_split(
    temporary_indices,
    test_size=0.5,
    random_state=random_state,
    stratify=temporary_targets
)

if (len(train_indices), len(validation_indices), len(test_indices)) != (16_000, 2_000, 2_000):
    raise RuntimeError('Unexpected split sizes')
if len(set(train_indices) | set(validation_indices) | set(test_indices)) != len(sample_indices):
    raise RuntimeError('Split indices overlap')

np.save(channels_path, training_channels)
metadata.to_csv(metadata_path, index=False)
np.savez(
    split_path,
    train_indices=train_indices,
    validation_indices=validation_indices,
    test_indices=test_indices
)

print('Complete')
print('Training channels:', training_channels.shape)
print('Metadata rows:', len(metadata))
print('Split rows:', len(train_indices), len(validation_indices), len(test_indices))
print('Output files:')
for output_path in output_paths:
    print('  ', output_path)
